In [1]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap

import pandas as pd
import numpy as np

In [2]:
prob_data = pd.read_csv("data_with_probs.csv")
combined_deep = pd.read_csv("combined_deep.csv")
input_deep = pd.read_csv("input_deep.csv")
output_deep = pd.read_csv("output_deep.csv")
data_adv = pd.read_csv("data_adv.csv")

/var/folders/55/cmm1r3h92wl2408nd3911m2c0000gn/T/ipykernel_60360/935775883.py:2: DtypeWarning: Columns (46) have mixed types. Specify dtype option on import or set low_memory=False.
  combined_deep = pd.read_csv("combined_deep.csv")
/var/folders/55/cmm1r3h92wl2408nd3911m2c0000gn/T/ipykernel_60360/935775883.py:3: DtypeWarning: Columns (46) have mixed types. Specify dtype option on import or set low_memory=False.
  input_deep = pd.read_csv("input_deep.csv")
/var/folders/55/cmm1r3h92wl2408nd3911m2c0000gn/T/ipykernel_60360/935775883.py:4: DtypeWarning: Columns (29) have mixed types. Specify dtype option on import or set low_memory=False.
  output_deep = pd.read_csv("output_deep.csv")


In [3]:
def draw_field(ax=None, figsize = (12, 6)):
    # Grab Axis Figure
    fig = ax.figure

    # Set Axis Limits
    ax.set_xlim(0, 120)
    ax.set_ylim(0, 53.3)

    # Add Rectangle for Field
    # --- Rectangle((bottom_left_x, bottom_left_y), width, height)
    ax.add_patch(Rectangle((0, 0), 120, 53.3, color="#3f995b"))

    # Add Rectangles for Endzones
    ax.add_patch(Rectangle((0, 0), 10, 53.3, color="#00338D"))
    ax.add_patch(Rectangle((110, 0), 10, 53.3, color="#00338D"))

    # Yardlines for every 5 yards
    for x in range(10, 111, 5):
        if x % 10 == 0: # make 10 yard lines bold
            linewidth = 2
        else:
            linewidth = 1

        # Plot yardlines - plot([start_x, end_x], [start_y, end_y]
        ax.plot([x, x], [0, 53.3], color = "white", linewidth = linewidth, alpha = 0.8)

    # Hash Marks - 70.75 feet from each sideline
    bottom_hash = 70.75 / 3
    top_hash = 53.3 - bottom_hash
    for x in range(11, 109):
        ax.plot([x, x], [bottom_hash - 0.2, bottom_hash + 0.2], color = "white")
        ax.plot([x, x], [top_hash - 0.2, top_hash + 0.2], color = "white")

    # Yardline Numbers every 10 yards
    for x in range(20, 110, 10):
        if x <= 60:
            yardline_num = x - 10
        else:
            yardline_num = 110 - x

        # Top yardline numbers
        ax.text(x + 1, 53.3 - 5, str(yardline_num), color="white", ha="center", va="center",
            fontsize=15, fontweight="bold", rotation=270, alpha=0.8)

        # Bottom yardline numbers
        ax.text(x - 1, 5, str(yardline_num), color="white", ha="center", va="center",
            fontsize=15, fontweight="bold", rotation=90, alpha=0.8)

    # Clean up axes
    ax.set_xticks([]) # remove all x-axis ticks
    ax.set_yticks([]) # remove all y-axis ticks
    ax.set_facecolor("#3f995b") # changes background of plotting to this color
    for spine in ax.spines.values(): # remove borders around plots
        spine.set_visible(False)

    return ax

def make_prob_plot(ax_prob, frames, cp_all_frames, first_after_frame):
    # Grab Axis Figure
    fig = ax_prob.figure

    prob_line, = ax_prob.plot(frames, cp_all_frames.values, color='blue', linewidth=2)
    indicator_line = ax_prob.axvline(frames[0], color='red', linestyle='--', linewidth=2)
    
    ax_prob.set_xlim(first_after_frame, frames[-1])
    ax_prob.set_ylim(0, 1)
    ax_prob.axhline(cp_all_frames.loc[first_after_frame], color='blue', linestyle='--', linewidth=1)
    ax_prob.set_xlabel("Frame Index")
    ax_prob.set_ylabel("Completion Probability")
    ax_prob.set_title("Completion Probability Over Time")
    
    return ax_prob, indicator_line

def get_play_attr(prob_data, attr):
    return prob_data.iloc[0][attr]

def make_play_context_str(prob_data):
    # Get Play Context
    complete = get_play_attr(prob_data, 'complete')
    possession_team = get_play_attr(prob_data, 'possession_team')
    home_team_abbr = get_play_attr(prob_data, 'home_team_abbr')
    visitor_team_abbr = get_play_attr(prob_data, 'visitor_team_abbr')
    pre_snap_home_score = get_play_attr(prob_data, 'pre_snap_home_score')
    pre_snap_visitor_score = get_play_attr(prob_data, 'pre_snap_visitor_score')
    quarter = get_play_attr(prob_data, 'quarter')
    game_clock = get_play_attr(prob_data, 'game_clock')
    down = get_play_attr(prob_data, 'down')
    yards_to_go = get_play_attr(prob_data, 'yards_to_go')
    pass_length = get_play_attr(prob_data, 'pass_length')
    route_of_targeted_receiver = get_play_attr(prob_data, 'route_of_targeted_receiver')
    team_coverage_type = get_play_attr(prob_data, 'team_coverage_type')
    complete = get_play_attr(prob_data, 'complete')
    tgt_rec_name = get_play_attr(prob_data, 'tgt_rec_name')
    yards_gained = get_play_attr(prob_data, 'yards_gained')

    # Calculate Net In-Air Completion Probability Change (Net In-Air CPC)
    cmp_prob_gained = (
        prob_data.groupby(['game_id', 'play_id'])['auc_slice']
                 .sum()
                 .reset_index()
    ).iloc[0]['auc_slice']

    # Set Play Context Text to Left of Prob Plot
    pos_score = pre_snap_home_score if possession_team == home_team_abbr else pre_snap_visitor_score
    def_score = pre_snap_home_score if possession_team == visitor_team_abbr else pre_snap_visitor_score
    def_team = home_team_abbr if possession_team == visitor_team_abbr else visitor_team_abbr
    complete_str = "complete" if complete == 1 else "incomplete"
    to_str = "to" if complete == 1 else "intended for"
    for_str = f"for {yards_gained} yards." if complete == 1 else ""
    
    if down == 1:
        down_suf = "st"
    elif down == 2:
        down_suf = "nd"
    elif down == 3:
        down_suf = "rd"
    else:
        down_suf = "th"

    # Set Context String
    context_str = (
        f"{possession_team}:{pos_score}\n"
        f"{def_team}:{def_score}\n"
        f"Q{quarter} {game_clock} {down}{down_suf} & {yards_to_go}\n"
        f"Route: {route_of_targeted_receiver}\n"
        f"Coverage: {team_coverage_type}\n"
        f"Pass Length: {pass_length}\n"
        f"Pass {complete_str} {to_str} {tgt_rec_name} {for_str}\n"
        f"Comp. Prob. Gained (after release): {cmp_prob_gained:.3f}\n"
    )

    return context_str

In [4]:
def animate_play(df, prob_data, game_id, play_id):
    
    # Get data for play only
    df_play = df[(df['game_id'] == game_id) & (df['play_id'] == play_id)].copy()
    prob_data = prob_data[(prob_data['game_id'] == game_id) & (prob_data['play_id'] == play_id)].copy()

    # Create List of 'frame_global' numbers
    frames = df_play['frame_global'].sort_values().unique()

    # Merge probability onto play frames
    prob_map = dict(zip(prob_data['frame_global'], prob_data['completion_prob']))
    df_play['completion_prob'] = df_play['frame_global'].map(prob_map).fillna(0)

    # Get 'frame_global' of first frame after ball release
    first_after_frame = df_play.loc[df_play['to_release'] == 'after', 'frame_global'].min()

    # Get ball landing locations
    ball_land_x = get_play_attr(prob_data, 'ball_land_x')
    ball_land_y = get_play_attr(prob_data, 'ball_land_y')

    # Setup figure with 2 panels
    fig = plt.figure(figsize=(16, 12))
    gs = gridspec.GridSpec(2, 3, height_ratios=[1, 1], width_ratios=[1, 1, 1], hspace=0.1, wspace=0.03)
    ax_field = fig.add_subplot(gs[0, :])
    ax_prob = fig.add_subplot(gs[1, 1])
    ax_context = fig.add_subplot(gs[1, 0])
    ax_context.axis("off")
    fig.subplots_adjust(left=0.01, right=0.99, top=0.90, bottom=0.05)

    # Draw field on field axis
    draw_field(ax_field)
    ax_field.set_aspect('equal')

    # Create DataFrame of Completion Prob. by Frame Global
    cp_all_frames = df_play[df_play['player_role']=='Targeted Receiver']\
                        .groupby('frame_global')['completion_prob']\
                        .first()\
                        .reindex(frames, fill_value=0)

    # Make Probability Plot
    ax_prob, indicator_line = make_prob_plot(ax_prob, frames, cp_all_frames, first_after_frame)

    # Scatter Plots for Players, Ball Landing
    player_scatter = ax_field.scatter([], [], s=60, color='white', edgecolor='black')
    rec_scatter = ax_field.scatter([], [], s=120, color='white', edgecolor='yellow', alpha=0.7)
    prob_text = ax_field.text(60, 55, "Completion Probability: 0.00", fontsize=20, color='black', ha='center')
    ax_field.scatter(ball_land_x, ball_land_y, s=60, color='saddlebrown', edgecolor='black', label='Ball Landing')

    # Make Play Context String, Display Play Context
    context_str = make_play_context_str(prob_data)
    ax_context.text(0.01, 0.99, context_str, fontsize=14, ha="left", va="top", transform=ax_context.transAxes)

    # Scale for size of receiver based on completion prob
    def receiver_size(prob):
        return 120 + 720 * prob

    # Create colormap to encode completion prob also with color
    rwb_cmap = LinearSegmentedColormap.from_list(
        "rwg_cmap",
        [(0, "red"), (0.5, "white"), (1, "blue")]
    )

    def update(frame_id):
        df_frame = df_play[df_play['frame_global'] == frame_id]

        # Non-Receivers
        df_non_rec = df_frame[df_frame['player_role'] != 'Targeted Receiver']
        player_scatter.set_offsets(np.column_stack([df_non_rec['x'], df_non_rec['y']]))

        # Receiver
        df_rec = df_frame[df_frame["player_role"] == 'Targeted Receiver']
        if len(df_rec) > 0:
            rec_scatter.set_offsets(np.column_stack([df_rec["x"], df_rec["y"]]))
        else:
            rec_scatter.set_offsets(np.empty((0, 2)))

        # Update probability text and indicator line
        curr_cp = df_frame[df_frame['player_role']=='Targeted Receiver']['completion_prob'].iloc[0] \
                  if len(df_rec) > 0 else 0

        # Adjust size/color of receiver based on completion probability
        rec_scatter.set_sizes([receiver_size(curr_cp)])
        rec_color = rwb_cmap(curr_cp)
        rec_scatter.set_facecolors([rec_color])
        
        prob_text.set_text(f"Completion Probability: {curr_cp:.2f}")
        indicator_line.set_xdata([frame_id, frame_id])

        return player_scatter, rec_scatter, indicator_line, prob_text

    anim = FuncAnimation(fig, update, frames=frames, interval=80, blit=False, repeat=False)
    plt.close(fig)

    return anim

In [6]:
anim = animate_play(combined_deep, prob_data, 2023101507, 2505)
HTML(anim.to_jshtml())

In [ ]:
filt = data_adv[(data_adv['game_id'] == 2023121007) &
                     (data_adv['play_id'] == 3240)].copy()

x = filt[['frame_global', 'flight_normalized_time', 'to_release', 'tgt_rec_x', 'tgt_rec_y', 'min_ball_dist_all_def']]
x.head(60)